# Notebook: Qualitative Evaluation (Experts 3, 4 & 5) -- Experiment 1

## Initial Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette("Set2")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

pd.set_option('display.notebook_repr_html', True)

def create_seaborn_boxplot(data, x, y, ax, title, ylabel, xlabel, scale_range=None):
    colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#ffd92f', '#e5c494', '#b3b3b3']
    unique_vals = sorted(data[x].unique())
    palette = colors[:len(unique_vals)]
    
    sns.boxplot(data=data, x=x, y=y, ax=ax, palette=palette,
                medianprops={'color': 'black', 'linewidth': 2.5, 'linestyle': ':'})
    
    for i, val in enumerate(unique_vals):
        mean_val = data[data[x] == val][y].mean()
        ax.scatter(i, mean_val, color='red', marker='D', s=50, zorder=3, 
                  edgecolor='darkred', linewidth=1)
    
    label_mapping = {
        'anthropic': 'Anthropic',
        'openai': 'OpenAI', 
        'google': 'Google',
        'deepseek': 'DeepSeek',
        'common': 'Common',
        'complex': 'Complex',
        'script': 'Script',
        'transcript': 'Transcript', 
        'tanenbaum': 'Tanenbaum',
        'script_manipulated': 'Script (Manipulated)',
        'exp1a': 'Exp 1a',
        'exp1b': 'Exp 1b'
    }
    
    current_labels = [tick.get_text() for tick in ax.get_xticklabels()]
    new_labels = [label_mapping.get(label, label) for label in current_labels]
    ax.set_xticklabels(new_labels, rotation=0)
    
    if scale_range:
        ax.set_ylim(scale_range)
        if scale_range == (0, 10):
            title += " (0-10 scale)"
        elif scale_range == (0, 70):
            title += " (0-70 scale)"
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=11, labelpad=15)
    ax.set_xlabel(xlabel, fontsize=11, labelpad=10)
    ax.grid(True, alpha=0.3)

BASE_PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

expert_base_path = os.path.join(BASE_PROJECT_PATH, "20_experiments/60_analyses/csv/qualitative/exp1/experts")
hint_base_path = os.path.join(BASE_PROJECT_PATH, "20_experiments/60_analyses/csv/qualitative/exp1/hints")

output_base_path = os.path.join(BASE_PROJECT_PATH, "40_evaluation/exp1/qualitative")
output_tables_path = os.path.join(output_base_path, "tables")
output_plots_path = os.path.join(output_base_path, "plots")

for path in [output_tables_path, output_plots_path]:
    os.makedirs(path, exist_ok=True)

tables = {}
plots = {}

print(f"Setup completed")
print(f"Output tables: {output_tables_path}")
print(f"Output plots: {output_plots_path}")

## Data Loading and Preprocessing

In [ ]:
def load_and_prepare_data():
    
    hint_exp1a = pd.read_csv(os.path.join(hint_base_path, "exp1a_hints.csv"))
    hint_exp1b = pd.read_csv(os.path.join(hint_base_path, "exp1b_hints.csv"))
    
    # Load all three experts data with proper CSV handling
    experts_data = {}
    for expert in [3, 4, 5]:
        expert_key = f'expert_{expert}'
        try:
            exp1a_path = os.path.join(expert_base_path, expert_key, "exp1a.csv")
            exp1b_path = os.path.join(expert_base_path, expert_key, "exp1b.csv")
            
            # Try different separators based on expert
            if expert == 4:
                # Expert 4 uses semicolons
                exp1a_data = pd.read_csv(exp1a_path, sep=';', quotechar='"')
                exp1b_data = pd.read_csv(exp1b_path, sep=';', quotechar='"')
            else:
                # Other experts use commas
                exp1a_data = pd.read_csv(exp1a_path, quotechar='"', escapechar='\\')
                exp1b_data = pd.read_csv(exp1b_path, quotechar='"', escapechar='\\')
            
            experts_data[expert_key] = {
                'exp1a': exp1a_data,
                'exp1b': exp1b_data
            }
            experts_data[expert_key]['exp1a']['expert'] = expert_key
            experts_data[expert_key]['exp1b']['expert'] = expert_key
            
            print(f"Loaded {expert_key}: Exp1a={len(experts_data[expert_key]['exp1a'])}, Exp1b={len(experts_data[expert_key]['exp1b'])}")
            
        except Exception as e:
            print(f"Error loading {expert_key}: {e}")
            continue
    
    if not experts_data:
        print("No expert data could be loaded!")
        return None, None, None, None, None
    
    # Combine all experts data
    exp1a_combined = pd.concat([data['exp1a'] for data in experts_data.values()], ignore_index=True)
    exp1b_combined = pd.concat([data['exp1b'] for data in experts_data.values()], ignore_index=True)
    
    # Add hint information - repeat hints for each expert
    hint_multiplier = len(experts_data)
    exp1a_hints_extended = pd.concat([hint_exp1a[['llm', 'prompt_type']]] * hint_multiplier, ignore_index=True)
    exp1b_hints_extended = pd.concat([hint_exp1b[['llm', 'prompt_type']]] * hint_multiplier, ignore_index=True)
    
    # Merge data carefully
    exp1a_df = exp1a_combined.copy()
    exp1b_df = exp1b_combined.copy()
    
    # Add missing columns from hints
    for col in ['llm', 'prompt_type']:
        if col not in exp1a_df.columns:
            exp1a_df[col] = exp1a_hints_extended[col].values
        if col not in exp1b_df.columns:
            exp1b_df[col] = exp1b_hints_extended[col].values
    
    numeric_cols_1a = ['relevance', 'clarity', 'answerability', 'challenging', 'value', 'language', 'correctness']
    numeric_cols_1b = ['relevance', 'clarity', 'answerability', 'challenging', 'value', 'language', 'manipulation_handling']
    
    # Clean and convert numeric columns to integers
    print("\nKonvertiere Bewertungen zu Integern...")
    for df, cols, exp_name in [(exp1a_df, numeric_cols_1a, "1a"), (exp1b_df, numeric_cols_1b, "1b")]:
        for col in cols:
            if col in df.columns:
                print(f"  Processing {exp_name}.{col}...")
                
                # Erstelle Debug-Info vor der Konvertierung
                non_numeric_before = df[col].apply(lambda x: not pd.isna(x) and not isinstance(x, (int, float))).sum()
                if non_numeric_before > 0:
                    print(f"    Found {non_numeric_before} non-numeric values before cleaning")
                
                # Bereinige ungültige Werte (aber behalte 0!)
                df[col] = df[col].replace(['??', '???', '', ' ', 'nan', 'NaN', 'NULL', 'null'], np.nan)
                
                # Konvertiere zu numeric (float first)
                df[col] = pd.to_numeric(df[col], errors='coerce')
                
                # Konvertiere zu Integer (behalte NaN bei)
                # Verwende 'Int64' dtype für nullable integers
                df[col] = df[col].astype('Int64')
                
                # Prüfe Wertebereich (0-10 für alle Kriterien)
                valid_data = df[col].dropna()
                if len(valid_data) > 0:
                    min_val = valid_data.min()
                    max_val = valid_data.max()
                    
                    # Prüfe erlaubten Bereich (0-10 für alle Kriterien)
                    valid_range = (0, 10)
                    
                    # Warnung bei Werten außerhalb des Bereichs
                    out_of_range = valid_data[(valid_data < valid_range[0]) | (valid_data > valid_range[1])]
                    if len(out_of_range) > 0:
                        print(f"    WARNING: {len(out_of_range)} values outside range {valid_range}")
                        # Entferne Werte außerhalb des gültigen Bereichs
                        df.loc[(df[col] < valid_range[0]) | (df[col] > valid_range[1]), col] = np.nan
                        df[col] = df[col].astype('Int64')
                    
                    print(f"    Range: {min_val}-{max_val}, Valid entries: {len(valid_data)}")
                else:
                    print(f"    No valid data found")
                
                # Debug: Prüfe finale Datentypen
                print(f"    Final dtype: {df[col].dtype}")
    
    exp1a_df['experiment'] = 'exp1a'
    exp1b_df['experiment'] = 'exp1b'
    
    # Handle input_source naming
    if 'input_source' in exp1b_df.columns:
        exp1b_df['input_source'] = exp1b_df['input_source'].replace('script', 'script_manipulated')
    
    # Calculate completion rates
    exp1a_filled = exp1a_df[numeric_cols_1a].notna().sum().sum()
    exp1a_total = len(exp1a_df) * len(numeric_cols_1a)
    exp1b_filled = exp1b_df[numeric_cols_1b].notna().sum().sum()
    exp1b_total = len(exp1b_df) * len(numeric_cols_1b)
    
    print(f"\nData loaded for {len(experts_data)} experts")
    print(f"Completion rates:")
    print(f"  Exp 1a: {exp1a_filled}/{exp1a_total} ({100*exp1a_filled/exp1a_total:.1f}%)")
    print(f"  Exp 1b: {exp1b_filled}/{exp1b_total} ({100*exp1b_filled/exp1b_total:.1f}%)")
    
    # Berechne total_score (verwende fillvalue=0 für Integer summation)
    exp1a_df['total_score'] = exp1a_df[numeric_cols_1a].sum(axis=1, skipna=True)
    exp1b_df['total_score'] = exp1b_df[numeric_cols_1b].sum(axis=1, skipna=True)
    
    # Zusätzliche Datentyp-Validierung für experts_data
    print(f"\nValidiere Datentypen in experts_data...")
    for expert_key in experts_data:
        for exp_type in ['exp1a', 'exp1b']:
            cols = numeric_cols_1a if exp_type == 'exp1a' else numeric_cols_1b
            for col in cols:
                if col in experts_data[expert_key][exp_type].columns:
                    # Bereinige auch die einzelnen Expert-DataFrames
                    experts_data[expert_key][exp_type][col] = experts_data[expert_key][exp_type][col].replace(['??', '???', '', ' ', 'nan', 'NaN', 'NULL', 'null'], np.nan)
                    experts_data[expert_key][exp_type][col] = pd.to_numeric(experts_data[expert_key][exp_type][col], errors='coerce')
                    
                    # Behalte 0-Werte für alle Kriterien!
                    experts_data[expert_key][exp_type][col] = experts_data[expert_key][exp_type][col].astype('Int64')
    
    print(f"Datentyp-Konvertierung abgeschlossen!")
    
    return exp1a_df, exp1b_df, numeric_cols_1a, numeric_cols_1b, experts_data

# Load data
result = load_and_prepare_data()
if result[0] is not None:
    exp1a_df, exp1b_df, numeric_cols_1a, numeric_cols_1b, experts_data = result
    
    print("\n" + "="*50)
    print("DATA OVERVIEW")
    print("="*50)
    
    for exp_name, exp_df in [("1a", exp1a_df), ("1b", exp1b_df)]:
        print(f"\nExperiment {exp_name}:")
        print(f"  Total samples: {len(exp_df)}")
        if 'expert' in exp_df.columns:
            print(f"  Experts: {dict(exp_df['expert'].value_counts())}")
        if 'llm' in exp_df.columns:
            print(f"  LLMs: {list(exp_df['llm'].unique())}")
        if 'prompt_type' in exp_df.columns:
            print(f"  Prompt Types: {list(exp_df['prompt_type'].unique())}")
        if 'input_source' in exp_df.columns:
            print(f"  Input Sources: {list(exp_df['input_source'].unique())}")
            
    # Zeige Datentypen für Überprüfung
    print(f"\n" + "="*50)
    print("DATENTYP ÜBERPRÜFUNG")
    print("="*50)
    
    print(f"\nExperiment 1a Datentypen:")
    for col in numeric_cols_1a:
        if col in exp1a_df.columns:
            print(f"  {col}: {exp1a_df[col].dtype}")
    
    print(f"\nExperiment 1b Datentypen:")
    for col in numeric_cols_1b:
        if col in exp1b_df.columns:
            print(f"  {col}: {exp1b_df[col].dtype}")
            
else:
    print("Failed to load data!")

# Experiment 1a: Original Content Analysis

Analysis of LLM question generation quality using original source materials (script, transcript, tanenbaum textbook).

## 1a.1: Descriptive Statistics

In [ ]:
print("EXPERIMENT 1A - DATENVALIDIERUNG UND DESCRIPTIVE STATISTICS")
print("="*60)

# Zusätzliche Datenvalidierung vor der Analyse
print("\nDATENVALIDIERUNG:")
print("="*30)

for exp_name, exp_df, cols in [("1a", exp1a_df, numeric_cols_1a), ("1b", exp1b_df, numeric_cols_1b)]:
    print(f"\nExperiment {exp_name}:")
    for col in cols:
        if col in exp_df.columns:
            # Prüfe Datentyp
            dtype = exp_df[col].dtype
            print(f"  {col}: {dtype}")
            
            # Prüfe auf ungültige Werte
            valid_data = exp_df[col].dropna()
            if len(valid_data) > 0:
                # Prüfe Wertebereich
                min_val = valid_data.min()
                max_val = valid_data.max()
                
                # Erwarteter Bereich: 0-10 für alle Kriterien
                expected_min = 0
                expected_max = 10
                
                if min_val < expected_min or max_val > expected_max:
                    print(f"    WARNING: Werte außerhalb Bereich [{expected_min}-{expected_max}]: {min_val}-{max_val}")
                else:
                    print(f"    ✓ Wertebereich korrekt: {min_val}-{max_val}")
                
                # Prüfe auf Nicht-Integer-Werte
                non_integer = valid_data[valid_data % 1 != 0]
                if len(non_integer) > 0:
                    print(f"    WARNING: {len(non_integer)} Nicht-Integer-Werte gefunden")
                    print(f"      Beispiele: {non_integer.head().tolist()}")
                else:
                    print(f"    ✓ Alle Werte sind Integer")
                    
                # Zeige Verteilung der Werte
                value_counts = valid_data.value_counts().sort_index()
                print(f"    Werteverteilung: {dict(value_counts.head(11))}")  # Zeige 0-10
            else:
                print(f"    Keine gültigen Daten")

print("\n" + "="*60)
print("DESCRIPTIVE STATISTICS")
print("="*60)

exp1a_stats = exp1a_df[numeric_cols_1a].describe().round(2)
tables['exp1a_overall_stats'] = exp1a_stats
print("\nOverall Statistics (all criteria):")
display(exp1a_stats)

exp1a_llm_stats = exp1a_df.groupby('llm')[numeric_cols_1a].agg(['mean', 'std', 'count']).round(2)
tables['exp1a_llm_stats'] = exp1a_llm_stats
print("\nStatistics by LLM:")
display(exp1a_llm_stats)

exp1a_source_stats = exp1a_df.groupby('input_source')[numeric_cols_1a].agg(['mean', 'std', 'count']).round(2)
tables['exp1a_source_stats'] = exp1a_source_stats
print("\nStatistics by Input Source:")
display(exp1a_source_stats)

exp1a_prompt_stats = exp1a_df.groupby('prompt_type')[numeric_cols_1a].agg(['mean', 'std', 'count']).round(2)
tables['exp1a_prompt_stats'] = exp1a_prompt_stats
print("\nStatistics by Prompt Type:")
display(exp1a_prompt_stats)

## 1a.2: Analysis by LLM

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 20))
axes = axes.flatten()
plots['exp1a_llm_analysis'] = fig

criteria = ['relevance', 'clarity', 'answerability', 'challenging', 'value', 'language', 'correctness']

for i, criterion in enumerate(criteria):
    create_seaborn_boxplot(exp1a_df, 'llm', criterion, axes[i], 
                          f'{criterion.title()}', criterion.title(), 'LLM', scale_range=(0, 10))

create_seaborn_boxplot(exp1a_df, 'llm', 'total_score', axes[7], 
                      'Total Score', 'Total Score', 'LLM', scale_range=(0, 70))

plt.suptitle('Experiment 1a: LLM Performance across all Criteria', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()

print("\n" + "="*80)
print("DETAILED STATISTICS - LLM PERFORMANCE (EXPERIMENT 1A)")
print("="*80)

all_criteria_with_total = criteria + ['total_score']
exp1a_llm_detailed_stats = exp1a_df.groupby('llm')[all_criteria_with_total].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables['exp1a_llm_detailed_stats'] = exp1a_llm_detailed_stats
print("\nDetailed Statistics by LLM (Experiment 1a):")
display(exp1a_llm_detailed_stats)

exp1a_llm_means = exp1a_df.groupby('llm')[numeric_cols_1a].mean().round(2)
tables['exp1a_llm_means'] = exp1a_llm_means
print("\nMean Scores by LLM (Experiment 1a):")
display(exp1a_llm_means)

exp1a_llm_overall = exp1a_llm_means.mean(axis=1).sort_values(ascending=False)
tables['exp1a_llm_ranking'] = exp1a_llm_overall
print("\nOverall LLM Ranking (Experiment 1a):")
for i, (llm, score) in enumerate(exp1a_llm_overall.items(), 1):
    print(f"{i}. {llm.title()}: {score:.2f}")

## 1a.3: Analysis by Input Source

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 20))
axes = axes.flatten()
plots['exp1a_source_analysis'] = fig

for i, criterion in enumerate(criteria):
    create_seaborn_boxplot(exp1a_df, 'input_source', criterion, axes[i], 
                          f'{criterion.title()}', criterion.title(), 'Input Source', scale_range=(0, 10))

create_seaborn_boxplot(exp1a_df, 'input_source', 'total_score', axes[7], 
                      'Total Score', 'Total Score', 'Input Source', scale_range=(0, 70))

plt.suptitle('Experiment 1a: Performance by Input Source', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()

print("\n" + "="*80)
print("DETAILED STATISTICS - INPUT SOURCE PERFORMANCE (EXPERIMENT 1A)")
print("="*80)

exp1a_source_detailed_stats = exp1a_df.groupby('input_source')[all_criteria_with_total].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables['exp1a_source_detailed_stats'] = exp1a_source_detailed_stats
print("\nDetailed Statistics by Input Source (Experiment 1a):")
display(exp1a_source_detailed_stats)

exp1a_source_means = exp1a_df.groupby('input_source')[numeric_cols_1a].mean().round(2)
tables['exp1a_source_means'] = exp1a_source_means
print("\nMean Scores by Input Source (Experiment 1a):")
display(exp1a_source_means)

exp1a_source_overall = exp1a_df.groupby('input_source')['total_score'].mean().sort_values(ascending=False)
tables['exp1a_source_ranking'] = exp1a_source_overall
print("\nInput Source Ranking by Total Score (Experiment 1a):")
for i, (source, score) in enumerate(exp1a_source_overall.items(), 1):
    print(f"{i}. {source.title()}: {score:.2f}/70")

# Total Score Paired Heatmaps
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
plots['exp1a_total_score_paired_heatmaps'] = fig

# Heatmap 1: Total Score by Input Source vs LLM
heatmap_mean = exp1a_df.groupby(['llm', 'input_source'])['total_score'].mean().unstack()
heatmap_std = exp1a_df.groupby(['llm', 'input_source'])['total_score'].std().unstack()

# Konvertiere zu float für seaborn-Kompatibilität
heatmap_mean = heatmap_mean.astype(float)
heatmap_std = heatmap_std.astype(float)

# Erstelle String-Matrix für Annotationen
annot_matrix = pd.DataFrame(index=heatmap_mean.index, columns=heatmap_mean.columns, dtype=str)
for i in range(len(heatmap_mean.index)):
    for j in range(len(heatmap_mean.columns)):
        mean_val = heatmap_mean.iloc[i, j]
        std_val = heatmap_std.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            annot_matrix.iloc[i, j] = f"{mean_val:.1f}\n({std_val:.1f})"
        elif pd.notna(mean_val):
            annot_matrix.iloc[i, j] = f"{mean_val:.1f}"
        else:
            annot_matrix.iloc[i, j] = ""

llm_labels = [label.title() for label in heatmap_mean.index]
source_labels = [label.title() for label in heatmap_mean.columns]

sns.heatmap(heatmap_mean, 
            annot=annot_matrix, 
            fmt='', 
            cmap='RdYlBu_r',
            center=heatmap_mean.mean().mean(),
            square=True,
            linewidths=0.5,
            linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Total Score'},
            annot_kws={'size': 10, 'weight': 'bold'},
            xticklabels=source_labels, 
            yticklabels=llm_labels, 
            ax=ax1)

ax1.set_title('Mean Total Score (LLM vs Input Source)\nValues: Mean (Std) | Scale: 0-70 points', 
             fontsize=14, fontweight='bold', pad=20)
ax1.set_xlabel('Input Source', fontsize=12, fontweight='bold', labelpad=10)
ax1.set_ylabel('LLM', fontsize=12, fontweight='bold', labelpad=10)

ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0, ha='center')
ax1.set_yticklabels(ax1.get_yticklabels(), rotation=0)

# Heatmap 2: Total Score by Prompt Type vs LLM
heatmap_mean_prompt = exp1a_df.groupby(['llm', 'prompt_type'])['total_score'].mean().unstack()
heatmap_std_prompt = exp1a_df.groupby(['llm', 'prompt_type'])['total_score'].std().unstack()

# Konvertiere zu float für seaborn-Kompatibilität
heatmap_mean_prompt = heatmap_mean_prompt.astype(float)
heatmap_std_prompt = heatmap_std_prompt.astype(float)

# Erstelle String-Matrix für Annotationen
annot_matrix_prompt = pd.DataFrame(index=heatmap_mean_prompt.index, columns=heatmap_mean_prompt.columns, dtype=str)
for i in range(len(heatmap_mean_prompt.index)):
    for j in range(len(heatmap_mean_prompt.columns)):
        mean_val = heatmap_mean_prompt.iloc[i, j]
        std_val = heatmap_std_prompt.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            annot_matrix_prompt.iloc[i, j] = f"{mean_val:.1f}\n({std_val:.1f})"
        elif pd.notna(mean_val):
            annot_matrix_prompt.iloc[i, j] = f"{mean_val:.1f}"
        else:
            annot_matrix_prompt.iloc[i, j] = ""

llm_labels = [label.title() for label in heatmap_mean_prompt.index]
prompt_labels = [label.title() for label in heatmap_mean_prompt.columns]

sns.heatmap(heatmap_mean_prompt, 
            annot=annot_matrix_prompt, 
            fmt='', 
            cmap='RdYlBu_r',
            center=heatmap_mean_prompt.mean().mean(),
            square=True,
            linewidths=0.5,
            linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Total Score'},
            annot_kws={'size': 10, 'weight': 'bold'},
            xticklabels=prompt_labels, 
            yticklabels=llm_labels, 
            ax=ax2)

ax2.set_title('Mean Total Score (LLM vs Prompt Type)\nValues: Mean (Std) | Scale: 0-70 points', 
             fontsize=14, fontweight='bold', pad=20)
ax2.set_xlabel('Prompt Type', fontsize=12, fontweight='bold', labelpad=10)
ax2.set_ylabel('LLM', fontsize=12, fontweight='bold', labelpad=10)

ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0, ha='center')
ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("DETAILED STATISTICS - LLM vs INPUT SOURCE COMBINATIONS (EXPERIMENT 1A)")
print("="*80)

exp1a_llm_source_combinations = exp1a_df.groupby(['llm', 'input_source'])['total_score'].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables['exp1a_llm_source_combinations'] = exp1a_llm_source_combinations
print("\nDetailed Statistics for LLM vs Input Source Combinations (Experiment 1a):")
display(exp1a_llm_source_combinations)

print("\n" + "="*80)
print("DETAILED STATISTICS - LLM vs PROMPT TYPE COMBINATIONS (EXPERIMENT 1A)")
print("="*80)

exp1a_llm_prompt_combinations = exp1a_df.groupby(['llm', 'prompt_type'])['total_score'].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables['exp1a_llm_prompt_combinations'] = exp1a_llm_prompt_combinations
print("\nDetailed Statistics for LLM vs Prompt Type Combinations (Experiment 1a):")
display(exp1a_llm_prompt_combinations)

## 1a.4: Analysis by Prompt Type

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()
plots['exp1a_prompt_analysis'] = fig

for i, criterion in enumerate(criteria):
    create_seaborn_boxplot(exp1a_df, 'prompt_type', criterion, axes[i], 
                          f'{criterion.title()}', criterion.title(), 'Prompt Type', scale_range=(0, 10))

create_seaborn_boxplot(exp1a_df, 'prompt_type', 'total_score', axes[7], 
                      'Total Score', 'Total Score', 'Prompt Type', scale_range=(0, 70))

plt.suptitle('Experiment 1a: Performance by Prompt Type', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

print("\n" + "="*80)
print("DETAILED STATISTICS - PROMPT TYPE PERFORMANCE (EXPERIMENT 1A)")
print("="*80)

exp1a_prompt_detailed_stats = exp1a_df.groupby('prompt_type')[all_criteria_with_total].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables['exp1a_prompt_detailed_stats'] = exp1a_prompt_detailed_stats
print("\nDetailed Statistics by Prompt Type (Experiment 1a):")
display(exp1a_prompt_detailed_stats)

exp1a_prompt_means = exp1a_df.groupby('prompt_type')[numeric_cols_1a].mean().round(2)
tables['exp1a_prompt_means'] = exp1a_prompt_means
print("\nMean Scores by Prompt Type (Experiment 1a):")
display(exp1a_prompt_means)

prompt_diff = exp1a_prompt_means.loc['complex'] - exp1a_prompt_means.loc['common']
print("\nComplex vs Common Prompt Difference (Experiment 1a):")
for criterion, diff in prompt_diff.items():
    direction = "higher" if diff > 0 else "lower"
    print(f"  {criterion.title()}: {diff:+.2f} ({direction} for complex)")

print("\n" + "="*80)
print("DETAILED STATISTICS - LLM vs PROMPT TYPE COMBINATIONS (EXPERIMENT 1A)")
print("="*80)

exp1a_llm_prompt_combinations = exp1a_df.groupby(['llm', 'prompt_type'])['total_score'].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables['exp1a_llm_prompt_combinations'] = exp1a_llm_prompt_combinations
print("\nDetailed Statistics for LLM vs Prompt Type Combinations (Experiment 1a):")
display(exp1a_llm_prompt_combinations)

In [ ]:
# Correctness-focused Heatmaps

print("\n" + "="*80)
print("CORRECTNESS ANALYSIS - PAIRED HEATMAPS")
print("="*80)

# Paired Heatmaps: Correctness by Input Source vs LLM and Prompt Type vs LLM
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
plots['exp1a_correctness_paired_heatmaps'] = fig

# Heatmap 1: Correctness by Input Source vs LLM
correctness_heatmap_mean = exp1a_df.groupby(['llm', 'input_source'])['correctness'].mean().unstack()
correctness_heatmap_std = exp1a_df.groupby(['llm', 'input_source'])['correctness'].std().unstack()

# Konvertiere zu float für seaborn-Kompatibilität
correctness_heatmap_mean = correctness_heatmap_mean.astype(float)
correctness_heatmap_std = correctness_heatmap_std.astype(float)

# Erstelle String-Matrix für Annotationen
correctness_annot_matrix = pd.DataFrame(index=correctness_heatmap_mean.index, columns=correctness_heatmap_mean.columns, dtype=str)
for i in range(len(correctness_heatmap_mean.index)):
    for j in range(len(correctness_heatmap_mean.columns)):
        mean_val = correctness_heatmap_mean.iloc[i, j]
        std_val = correctness_heatmap_std.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            correctness_annot_matrix.iloc[i, j] = f"{mean_val:.1f}\n({std_val:.1f})"
        elif pd.notna(mean_val):
            correctness_annot_matrix.iloc[i, j] = f"{mean_val:.1f}"
        else:
            correctness_annot_matrix.iloc[i, j] = ""

llm_labels = [label.title() for label in correctness_heatmap_mean.index]
source_labels = [label.title() for label in correctness_heatmap_mean.columns]

sns.heatmap(correctness_heatmap_mean, 
            annot=correctness_annot_matrix, 
            fmt='', 
            cmap='RdYlBu_r',
            center=correctness_heatmap_mean.mean().mean(),
            square=True,
            linewidths=0.5,
            linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Correctness Score'},
            annot_kws={'size': 10, 'weight': 'bold'},
            xticklabels=source_labels, 
            yticklabels=llm_labels, 
            ax=ax1)

ax1.set_title('Correctness Score (LLM vs Input Source)\nValues: Mean (Std) | Scale: 0-10 points', 
             fontsize=14, fontweight='bold', pad=20)
ax1.set_xlabel('Input Source', fontsize=12, fontweight='bold', labelpad=10)
ax1.set_ylabel('LLM', fontsize=12, fontweight='bold', labelpad=10)

ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0, ha='center')
ax1.set_yticklabels(ax1.get_yticklabels(), rotation=0)

# Heatmap 2: Correctness by Prompt Type vs LLM
correctness_prompt_heatmap_mean = exp1a_df.groupby(['llm', 'prompt_type'])['correctness'].mean().unstack()
correctness_prompt_heatmap_std = exp1a_df.groupby(['llm', 'prompt_type'])['correctness'].std().unstack()

# Konvertiere zu float für seaborn-Kompatibilität
correctness_prompt_heatmap_mean = correctness_prompt_heatmap_mean.astype(float)
correctness_prompt_heatmap_std = correctness_prompt_heatmap_std.astype(float)

# Erstelle String-Matrix für Annotationen
correctness_prompt_annot_matrix = pd.DataFrame(index=correctness_prompt_heatmap_mean.index, columns=correctness_prompt_heatmap_mean.columns, dtype=str)
for i in range(len(correctness_prompt_heatmap_mean.index)):
    for j in range(len(correctness_prompt_heatmap_mean.columns)):
        mean_val = correctness_prompt_heatmap_mean.iloc[i, j]
        std_val = correctness_prompt_heatmap_std.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            correctness_prompt_annot_matrix.iloc[i, j] = f"{mean_val:.1f}\n({std_val:.1f})"
        elif pd.notna(mean_val):
            correctness_prompt_annot_matrix.iloc[i, j] = f"{mean_val:.1f}"
        else:
            correctness_prompt_annot_matrix.iloc[i, j] = ""

llm_labels = [label.title() for label in correctness_prompt_heatmap_mean.index]
prompt_labels = [label.title() for label in correctness_prompt_heatmap_mean.columns]

sns.heatmap(correctness_prompt_heatmap_mean, 
            annot=correctness_prompt_annot_matrix, 
            fmt='', 
            cmap='RdYlBu_r',
            center=correctness_prompt_heatmap_mean.mean().mean(),
            square=True,
            linewidths=0.5,
            linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Correctness Score'},
            annot_kws={'size': 10, 'weight': 'bold'},
            xticklabels=prompt_labels, 
            yticklabels=llm_labels, 
            ax=ax2)

ax2.set_title('Correctness Score (LLM vs Prompt Type)\nValues: Mean (Std) | Scale: 0-10 points', 
             fontsize=14, fontweight='bold', pad=20)
ax2.set_xlabel('Prompt Type', fontsize=12, fontweight='bold', labelpad=10)
ax2.set_ylabel('LLM', fontsize=12, fontweight='bold', labelpad=10)

ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0, ha='center')
ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.show()

# Statistics for both heatmaps
correctness_llm_source_stats = exp1a_df.groupby(['llm', 'input_source'])['correctness'].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables['exp1a_correctness_llm_source_stats'] = correctness_llm_source_stats
print("\nDetailed Correctness Statistics by LLM and Input Source:")
display(correctness_llm_source_stats)

correctness_llm_prompt_stats = exp1a_df.groupby(['llm', 'prompt_type'])['correctness'].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables['exp1a_correctness_llm_prompt_stats'] = correctness_llm_prompt_stats
print("\nDetailed Correctness Statistics by LLM and Prompt Type:")
display(correctness_llm_prompt_stats)

In [ ]:
print("\n" + "="*80)
print("CORRECTNESS INSIGHTS")
print("="*80)

# Best LLM for correctness
correctness_by_llm = exp1a_df.groupby('llm')['correctness'].mean().sort_values(ascending=False)
print(f"\nCorrectness Ranking by LLM:")
for i, (llm, score) in enumerate(correctness_by_llm.items(), 1):
    print(f"{i}. {llm.title()}: {score:.2f}")

# Best input source for correctness
correctness_by_source = exp1a_df.groupby('input_source')['correctness'].mean().sort_values(ascending=False)
print(f"\nCorrectness Ranking by Input Source:")
for i, (source, score) in enumerate(correctness_by_source.items(), 1):
    print(f"{i}. {source.title()}: {score:.2f}")

# Best prompt type for correctness
correctness_by_prompt = exp1a_df.groupby('prompt_type')['correctness'].mean().sort_values(ascending=False)
print(f"\nCorrectness Ranking by Prompt Type:")
for i, (prompt, score) in enumerate(correctness_by_prompt.items(), 1):
    print(f"{i}. {prompt.title()}: {score:.2f}")

# Identify best combinations
best_llm_source = exp1a_df.groupby(['llm', 'input_source'])['correctness'].mean().idxmax()
best_llm_source_score = exp1a_df.groupby(['llm', 'input_source'])['correctness'].mean().max()
print(f"\nBest LLM-Source combination for correctness:")
print(f"  {best_llm_source[0].title()} + {best_llm_source[1].title()}: {best_llm_source_score:.2f}")

best_llm_prompt = exp1a_df.groupby(['llm', 'prompt_type'])['correctness'].mean().idxmax()
best_llm_prompt_score = exp1a_df.groupby(['llm', 'prompt_type'])['correctness'].mean().max()
print(f"\nBest LLM-Prompt combination for correctness:")
print(f"  {best_llm_prompt[0].title()} + {best_llm_prompt[1].title()}: {best_llm_prompt_score:.2f}")

# Experiment 1b: Manipulated Content Analysis

In [ ]:
print("EXPERIMENT 1B - DATA AVAILABILITY CHECK")
print("="*60)

exp1b_filled = exp1b_df[numeric_cols_1b].notna().sum().sum()
exp1b_total = len(exp1b_df) * len(numeric_cols_1b)

print(f"Exp 1b completion: {exp1b_filled}/{exp1b_total} ({100*exp1b_filled/exp1b_total:.1f}%)")

if exp1b_filled > 0:
    print("Proceeding with Experiment 1b analysis")
    analyze_exp1b = True
else:
    print("No ratings available for Experiment 1b yet - showing structure only")
    analyze_exp1b = False
    
print(f"\nExperiment 1b structure:")
print(f"  Samples: {len(exp1b_df)}")
print(f"  LLMs: {list(exp1b_df['llm'].unique())}")
print(f"  Prompt Types: {list(exp1b_df['prompt_type'].unique())}")
print(f"  Input Sources: {list(exp1b_df['input_source'].unique())}")
print(f"  Evaluation Criteria: {numeric_cols_1b}")

print(f"\nSample Exp1b structure:")
display(exp1b_df[['input_source', 'layer', 'llm', 'prompt_type'] + numeric_cols_1b].head())

In [ ]:
if analyze_exp1b:
    print("\nEXPERIMENT 1B - DESCRIPTIVE STATISTICS")
    print("="*60)
    
    exp1b_stats = exp1b_df[numeric_cols_1b].describe().round(2)
    tables['exp1b_overall_stats'] = exp1b_stats
    print("\nOverall Statistics (all criteria):")
    display(exp1b_stats)
    
    exp1b_llm_stats = exp1b_df.groupby('llm')[numeric_cols_1b].agg(['mean', 'std', 'count']).round(2)
    tables['exp1b_llm_stats'] = exp1b_llm_stats
    print("\nStatistics by LLM:")
    display(exp1b_llm_stats)
    
    exp1b_prompt_stats = exp1b_df.groupby('prompt_type')[numeric_cols_1b].agg(['mean', 'std', 'count']).round(2)
    tables['exp1b_prompt_stats'] = exp1b_prompt_stats
    print("\nStatistics by Prompt Type:")
    display(exp1b_prompt_stats)
    
else:
    print("\nExperiment 1b analysis will be available once expert evaluations are completed.")
    print("The structure is ready for analysis of manipulation handling capabilities.")

In [ ]:
if analyze_exp1b:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    plots['exp1b_llm_analysis'] = fig
    
    criteria_1b = ['relevance', 'clarity', 'answerability', 'challenging', 'value', 'language', 'manipulation_handling']
    
    for i, criterion in enumerate(criteria_1b):
        create_seaborn_boxplot(exp1b_df, 'llm', criterion, axes[i], 
                              f'{criterion.title()}', criterion.title(), 'LLM', scale_range=(0, 10))
    
    create_seaborn_boxplot(exp1b_df, 'llm', 'total_score', axes[7], 
                          'Total Score', 'Total Score', 'LLM', scale_range=(0, 70))
    
    plt.suptitle('Experiment 1b: LLM Performance (Manipulation Handling)', fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.show()
    
    print("\n" + "="*80)
    print("DETAILED STATISTICS - LLM PERFORMANCE (EXPERIMENT 1B)")
    print("="*80)
    
    all_criteria_1b_with_total = criteria_1b + ['total_score']
    exp1b_llm_detailed_stats = exp1b_df.groupby('llm')[all_criteria_1b_with_total].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
    tables['exp1b_llm_detailed_stats'] = exp1b_llm_detailed_stats
    print("\nDetailed Statistics by LLM (Experiment 1b):")
    display(exp1b_llm_detailed_stats)
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    plots['exp1b_prompt_analysis'] = fig
    
    for i, criterion in enumerate(criteria_1b):
        create_seaborn_boxplot(exp1b_df, 'prompt_type', criterion, axes[i], 
                              f'{criterion.title()}', criterion.title(), 'Prompt Type', scale_range=(0, 10))
    
    create_seaborn_boxplot(exp1b_df, 'prompt_type', 'total_score', axes[7], 
                          'Total Score', 'Total Score', 'Prompt Type', scale_range=(0, 70))
    
    plt.suptitle('Experiment 1b: Performance by Prompt Type (Manipulation Handling)', fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.show()
    
    print("\n" + "="*80)
    print("DETAILED STATISTICS - PROMPT TYPE PERFORMANCE (EXPERIMENT 1B)")
    print("="*80)
    
    exp1b_prompt_detailed_stats = exp1b_df.groupby('prompt_type')[all_criteria_1b_with_total].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
    tables['exp1b_prompt_detailed_stats'] = exp1b_prompt_detailed_stats
    print("\nDetailed Statistics by Prompt Type (Experiment 1b):")
    display(exp1b_prompt_detailed_stats)
    
    exp1b_llm_means = exp1b_df.groupby('llm')[numeric_cols_1b].mean().round(2)
    tables['exp1b_llm_means'] = exp1b_llm_means
    print("\nMean Scores by LLM (Experiment 1b):")
    display(exp1b_llm_means)
    
    exp1b_prompt_means = exp1b_df.groupby('prompt_type')[numeric_cols_1b].mean().round(2)
    tables['exp1b_prompt_means'] = exp1b_prompt_means
    print("\nMean Scores by Prompt Type (Experiment 1b):")
    display(exp1b_prompt_means)
    
else:
    print("Visualization section ready for Experiment 1b data")
    print("Will include: LLM analysis, Prompt type comparison, Manipulation handling focus")

# Summary and Key Insights

This section provides a comprehensive summary of findings from the qualitative analysis of Experiment 1.

## Inter-Rater Agreement Analysis

In [ ]:
# Kendalls W Berechnung für Experten 3, 4, 5 (ERSETZT FLEISS KAPPA)
# Kendalls W ist für ordinale Daten (0-10) und Übereinstimmung zwischen mehreren Ratern geeignet

import numpy as np
import pandas as pd
import kendall_w as kw  # Korrekte Import-Syntax

def kendall_w_level(w):
    """Interpretation von Kendalls W Werten"""
    if w < 0.1:
        return "Very Weak"
    elif w < 0.3:
        return "Weak"
    elif w < 0.5:
        return "Moderate" 
    elif w < 0.7:
        return "Strong"
    else:
        return "Very Strong"

def calculate_kendalls_w_for_criterion(expert3_df, expert4_df, expert5_df, criterion):
    """
    Berechnet Kendalls W für ein spezifisches Kriterium zwischen den drei Experten.
    
    Args:
        expert3_df, expert4_df, expert5_df: DataFrames der einzelnen Experten
        criterion: Das zu analysierende Kriterium (z.B. 'relevance')
    
    Returns:
        dict: Enthält W-Wert, Agreement Level und weitere Statistiken
    """
    # Debug: Zeige verfügbare Spalten
    print(f"DEBUG: Verfügbare Spalten für {criterion}:")
    print(f"Expert 3: {list(expert3_df.columns)}")
    
    # Verwende nur input_source, layer und sample_id als Merge-Schlüssel
    merge_cols = ['input_source', 'layer', 'sample_id']
    
    # Prüfe ob alle Merge-Spalten vorhanden sind
    missing_cols = [col for col in merge_cols if col not in expert3_df.columns]
    if missing_cols:
        print(f"  Fehlende Spalten: {missing_cols}")
        # Fallback: verwende nur verfügbare Spalten
        merge_cols = [col for col in merge_cols if col in expert3_df.columns]
        if not merge_cols:
            # Als letzter Ausweg: verwende Index-basiertes Merging
            merge_cols = None
    
    try:
        if merge_cols:
            # Merge die Daten aller drei Experten basierend auf verfügbaren Spalten
            merged_data = expert3_df[merge_cols + [criterion]].copy()
            merged_data = merged_data.rename(columns={criterion: 'expert_3'})
            
            # Füge Expert 4 hinzu
            exp4_data = expert4_df[merge_cols + [criterion]].copy()
            exp4_data = exp4_data.rename(columns={criterion: 'expert_4'})
            merged_data = merged_data.merge(exp4_data, on=merge_cols, how='inner')
            
            # Füge Expert 5 hinzu
            exp5_data = expert5_df[merge_cols + [criterion]].copy()
            exp5_data = exp5_data.rename(columns={criterion: 'expert_5'})
            merged_data = merged_data.merge(exp5_data, on=merge_cols, how='inner')
        else:
            # Index-basiertes Merging als Fallback
            print(f"  Verwende Index-basiertes Merging für {criterion}")
            expert3_ratings = expert3_df[criterion].reset_index(drop=True)
            expert4_ratings = expert4_df[criterion].reset_index(drop=True)
            expert5_ratings = expert5_df[criterion].reset_index(drop=True)
            
            # Stelle sicher, dass alle Arrays die gleiche Länge haben
            min_length = min(len(expert3_ratings), len(expert4_ratings), len(expert5_ratings))
            expert3_ratings = expert3_ratings[:min_length]
            expert4_ratings = expert4_ratings[:min_length]
            expert5_ratings = expert5_ratings[:min_length]
            
            merged_data = pd.DataFrame({
                'expert_3': expert3_ratings,
                'expert_4': expert4_ratings,
                'expert_5': expert5_ratings
            })
        
        # Entferne Zeilen mit NaN-Werten
        merged_data = merged_data.dropna(subset=['expert_3', 'expert_4', 'expert_5'])
        
        if len(merged_data) < 3:
            return {
                'Criterion': criterion.title(),
                'Kendalls_W': np.nan,
                'Agreement_Level': 'Insufficient Data',
                'N_Items': len(merged_data),
                'Mean_Rating': np.nan,
                'Std_Rating': np.nan,
                'Example_Ratings': 'N/A'
            }
        
        # Bereite die Daten für Kendalls W vor (Items x Raters Matrix)
        # Format: Liste von Listen, wobei jede Subliste die Bewertungen aller Rater für ein Item enthält
        annotations = []
        for idx, row in merged_data.iterrows():
            annotations.append([
                int(row['expert_3']) if not pd.isna(row['expert_3']) else 0,
                int(row['expert_4']) if not pd.isna(row['expert_4']) else 0,
                int(row['expert_5']) if not pd.isna(row['expert_5']) else 0
            ])
        
        # Berechne Kendalls W mit korrekter Syntax
        w_value = kw.compute_w(annotations)
        
        # Berechne zusätzliche Statistiken
        ratings_matrix = np.array(annotations)
        mean_rating = np.mean(ratings_matrix)
        std_rating = np.std(ratings_matrix)
        
        # Erstelle Beispiel-Ratings für Output
        example_count = min(3, len(annotations))
        example_ratings = annotations[:example_count]
        
        return {
            'Criterion': criterion.title(),
            'Kendalls_W': round(w_value, 3),
            'Agreement_Level': kendall_w_level(w_value),
            'N_Items': len(merged_data),
            'Mean_Rating': round(mean_rating, 2),
            'Std_Rating': round(std_rating, 2),
            'Example_Ratings': str(example_ratings)
        }
        
    except Exception as e:
        print(f"Error calculating Kendalls W for {criterion}: {e}")
        return {
            'Criterion': criterion.title(),
            'Kendalls_W': np.nan,
            'Agreement_Level': 'Calculation Error',
            'N_Items': 0,
            'Mean_Rating': np.nan,
            'Std_Rating': np.nan,
            'Example_Ratings': str(e)
        }

def simple_kendalls_w(expert3_df, expert4_df, expert5_df, criteria_cols):
    """
    Berechnet Kendalls W für alle Kriterien zwischen den Experten 3, 4, 5.
    
    Args:
        expert3_df, expert4_df, expert5_df: DataFrames der einzelnen Experten
        criteria_cols: Liste der zu analysierenden Kriterien
    
    Returns:
        DataFrame: Enthält W-Werte und Statistiken für alle Kriterien
    """
    print("DEBUG: Verfügbare Spalten in Experten-Daten:")
    print()
    
    for i, (expert_key, expert_df) in enumerate([('expert_3', expert3_df), ('expert_4', expert4_df), ('expert_5', expert5_df)], 3):
        print(f"{expert_key} exp1a Spalten:")
        print(list(expert_df.columns))
        print(f"Shape: {expert_df.shape}")
        print()
    
    print("=" * 60)
    print("KENDALLS W ANALYSE - VEREINFACHT")
    print("=" * 60)
    
    results = []
    
    for criterion in criteria_cols:
        print(f"Berechne Kendalls W für: {criterion}")
        
        result = calculate_kendalls_w_for_criterion(expert3_df, expert4_df, expert5_df, criterion)
        results.append(result)
        
        # Ausgabe der Ergebnisse
        w_val = result['Kendalls_W']
        level = result['Agreement_Level']
        n_items = result['N_Items']
        
        if not pd.isna(w_val):
            print(f"  Kendalls W: {w_val:.3f} ({level})")
            print(f"  Items analysiert: {n_items}")
        else:
            print(f"  Status: {level}")
        print()
    
    return pd.DataFrame(results)

In [ ]:
# Debug: Schaue die verfügbaren Spalten an
print("DEBUG: Verfügbare Spalten in Experten-Daten:")
for expert_key in ['expert_3', 'expert_4', 'expert_5']:
    if expert_key in experts_data:
        print(f"\n{expert_key} exp1a Spalten:")
        print(list(experts_data[expert_key]['exp1a'].columns))
        print(f"Shape: {experts_data[expert_key]['exp1a'].shape}")

# Vereinfachte Kendalls W Berechnung nur mit den verfügbaren Spalten
def calculate_kendalls_w_simple(expert3_df, expert4_df, expert5_df, criterion):
    """
    Vereinfachte Kendalls W Berechnung ohne Merge auf zusätzliche Spalten
    """
    # Prüfe ob das Kriterium in allen DataFrames existiert
    if criterion not in expert3_df.columns or criterion not in expert4_df.columns or criterion not in expert5_df.columns:
        return {
            'Criterion': criterion.title(),
            'Kendalls_W': np.nan,
            'P_Value': np.nan,
            'Agreement_Level': 'Column Missing',
            'N_Items': 0,
            'Mean_Rating': np.nan,
            'Std_Rating': np.nan,
            'Example_Ratings': 'N/A'
        }
    
    # Nimm nur die ersten N Items, wo alle Experten Bewertungen haben
    min_len = min(len(expert3_df), len(expert4_df), len(expert5_df))
    
    # Extrahiere die Bewertungen für das Kriterium
    exp3_ratings = expert3_df[criterion].iloc[:min_len].dropna()
    exp4_ratings = expert4_df[criterion].iloc[:min_len].dropna()
    exp5_ratings = expert5_df[criterion].iloc[:min_len].dropna()
    
    # Finde Items wo alle Experten Bewertungen haben
    valid_indices = []
    all_ratings = []
    
    for i in range(min_len):
        if (i < len(exp3_ratings) and i < len(exp4_ratings) and i < len(exp5_ratings) and
            not pd.isna(expert3_df[criterion].iloc[i]) and 
            not pd.isna(expert4_df[criterion].iloc[i]) and 
            not pd.isna(expert5_df[criterion].iloc[i])):
            
            ratings_row = [
                expert3_df[criterion].iloc[i],
                expert4_df[criterion].iloc[i],
                expert5_df[criterion].iloc[i]
            ]
            all_ratings.append(ratings_row)
            valid_indices.append(i)
    
    if len(all_ratings) < 3:
        return {
            'Criterion': criterion.title(),
            'Kendalls_W': np.nan,
            'P_Value': np.nan,
            'Agreement_Level': 'Insufficient Data',
            'N_Items': len(all_ratings),
            'Mean_Rating': np.nan,
            'Std_Rating': np.nan,
            'Example_Ratings': 'N/A'
        }
    
    # Konvertiere zu NumPy Array
    ratings_matrix = np.array(all_ratings)
    
    # Berechne Kendalls W
    try:
        w_value, p_value = kendall_w(ratings_matrix)
        
        # Berechne zusätzliche Statistiken
        mean_rating = np.mean(ratings_matrix)
        std_rating = np.std(ratings_matrix)
        
        # Erstelle Beispiel-Ratings
        example_indices = np.random.choice(len(all_ratings), min(3, len(all_ratings)), replace=False)
        example_ratings = [all_ratings[i] for i in example_indices]
        
        return {
            'Criterion': criterion.title(),
            'Kendalls_W': round(w_value, 3),
            'P_Value': round(p_value, 3) if not np.isnan(p_value) else np.nan,
            'Agreement_Level': kendall_w_level(w_value),
            'N_Items': len(all_ratings),
            'Mean_Rating': round(mean_rating, 2),
            'Std_Rating': round(std_rating, 2),
            'Example_Ratings': str(example_ratings)
        }
        
    except Exception as e:
        print(f"Error calculating Kendalls W for {criterion}: {e}")
        return {
            'Criterion': criterion.title(),
            'Kendalls_W': np.nan,
            'P_Value': np.nan,
            'Agreement_Level': 'Calculation Error',
            'N_Items': len(all_ratings),
            'Mean_Rating': round(np.mean(ratings_matrix), 2),
            'Std_Rating': round(np.std(ratings_matrix), 2),
            'Example_Ratings': str(e)
        }

# Führe vereinfachte Kendalls W Analyse durch
print("\n" + "="*60)
print("KENDALLS W ANALYSE - VEREINFACHT")
print("="*60)

results = []
for criterion in numeric_cols_1a:
    print(f"Berechne Kendalls W für: {criterion}")
    
    result = calculate_kendalls_w_simple(
        experts_data['expert_3']['exp1a'],
        experts_data['expert_4']['exp1a'],
        experts_data['expert_5']['exp1a'],
        criterion
    )
    results.append(result)
    
    # Ausgabe der Ergebnisse
    w_val = result['Kendalls_W']
    level = result['Agreement_Level']
    n_items = result['N_Items']
    
    if not np.isnan(w_val):
        print(f"  Kendalls W: {w_val:.3f} ({level})")
        print(f"  Items analysiert: {n_items}")
    else:
        print(f"  Status: {level}")
    print()

kendalls_w_results = pd.DataFrame(results)

# Speichere die Ergebnisse
tables['kendalls_w_exp1a_experts35'] = kendalls_w_results

print("\nKendalls W Ergebnisse (Experiment 1a, Experten 3-5):")
display(kendalls_w_results)

# Zusätzliche Analyse
valid_w_values = kendalls_w_results['Kendalls_W'].dropna()
if len(valid_w_values) > 0:
    avg_w = valid_w_values.mean()
    print(f"\nZusammenfassung der Übereinstimmung:")
    print(f"Durchschnittliches Kendalls W: {avg_w:.3f}")
    print(f"Übereinstimmungslevel: {kendall_w_level(avg_w)}")
    print(f"Interpretation: Kendalls W misst die Rangordnungs-Übereinstimmung bei ordinalen Daten")
    
    # Zeige beste und schlechteste Übereinstimmung
    if len(valid_w_values) > 1:
        best_criterion = kendalls_w_results.loc[kendalls_w_results['Kendalls_W'].idxmax()]
        worst_criterion = kendalls_w_results.loc[kendalls_w_results['Kendalls_W'].idxmin()]
        
        print(f"\nBeste Übereinstimmung: {best_criterion['Criterion']} (W = {best_criterion['Kendalls_W']:.3f}, {best_criterion['Agreement_Level']})")
        print(f"Schlechteste Übereinstimmung: {worst_criterion['Criterion']} (W = {worst_criterion['Kendalls_W']:.3f}, {worst_criterion['Agreement_Level']})")
else:
    print("\nKeine gültigen Kendalls W Werte berechnet.")

In [ ]:
# Test der Kendalls W Implementierung mit einem einfachen Beispiel
import kendall_w as kw

# Test mit dem Beispiel aus der Dokumentation
test_annotations = [
    [1, 1, 1, 2],
    [2, 2, 2, 3],
    [3, 3, 3, 1],
]

print("Test der kendall_w Bibliothek:")
print("Beispiel annotations:", test_annotations)

try:
    test_w = kw.compute_w(test_annotations)
    print(f"Kendalls W Ergebnis: {test_w:.4f}")
    print("✓ kendall_w funktioniert korrekt!")
except Exception as e:
    print(f"Fehler bei kendall_w: {e}")

# Teste mit unseren echten Daten
print("\nTeste mit echten Expertendaten:")
try:
    # Hole die ersten 5 Bewertungen für 'relevance' von allen drei Experten
    sample_data = []
    
    for i in range(min(5, len(experts_data['expert_3']['exp1a']))):
        expert3_rating = experts_data['expert_3']['exp1a']['relevance'].iloc[i]
        expert4_rating = experts_data['expert_4']['exp1a']['relevance'].iloc[i]
        expert5_rating = experts_data['expert_5']['exp1a']['relevance'].iloc[i]
        
        # Nur Zeilen mit gültigen Daten verwenden
        if pd.notna(expert3_rating) and pd.notna(expert4_rating) and pd.notna(expert5_rating):
            sample_data.append([int(expert3_rating), int(expert4_rating), int(expert5_rating)])
    
    if sample_data:
        print(f"Sample data: {sample_data}")
        sample_w = kw.compute_w(sample_data)
        print(f"Kendalls W für 'relevance' Sample: {sample_w:.4f}")
        print("✓ Echte Daten funktionieren!")
    else:
        print("Keine gültigen Sample-Daten gefunden")
        
except Exception as e:
    print(f"Fehler mit echten Daten: {e}")

In [ ]:
# Vollständige Kendalls W Analyse für alle Kriterien
def calculate_kendalls_w_corrected(expert3_df, expert4_df, expert5_df, criteria_cols):
    """
    Berechnet Kendalls W für alle Kriterien zwischen den drei Experten.
    Verwendet die korrekte kendall_w.compute_w() Syntax.
    """
    print("KENDALLS W für Experten 3, 4, 5 - Experiment 1a")
    print("="*60)
    print("Kendalls W misst die Übereinstimmung zwischen mehreren Ratern bei ordinalen Daten (0-10).")
    print("W-Werte näher bei 1 zeigen höhere Übereinstimmung an.")
    print()
    
    results = []
    
    for criterion in criteria_cols:
        print(f"Berechne Kendalls W für: {criterion}")
        
        try:
            # Sammle alle gültigen Bewertungen für dieses Kriterium
            annotations = []
            
            # Durchlaufe alle Zeilen und sammle Bewertungen
            min_len = min(len(expert3_df), len(expert4_df), len(expert5_df))
            
            for i in range(min_len):
                expert3_rating = expert3_df[criterion].iloc[i]
                expert4_rating = expert4_df[criterion].iloc[i]
                expert5_rating = expert5_df[criterion].iloc[i]
                
                # Nur Zeilen mit allen gültigen Bewertungen verwenden
                if (pd.notna(expert3_rating) and pd.notna(expert4_rating) and 
                    pd.notna(expert5_rating)):
                    annotations.append([
                        int(expert3_rating), 
                        int(expert4_rating), 
                        int(expert5_rating)
                    ])
            
            if len(annotations) >= 3:  # Mindestens 3 Items für aussagekräftige Analyse
                # Berechne Kendalls W
                w_value = kw.compute_w(annotations)
                
                # Berechne zusätzliche Statistiken
                ratings_array = np.array(annotations)
                mean_rating = np.mean(ratings_array)
                std_rating = np.std(ratings_array)
                
                # Erstelle Beispiel-Ratings (erste 3)
                example_ratings = annotations[:3]
                
                # Klassifikation der Übereinstimmung
                if w_value < 0.1:
                    agreement_level = "Very Weak"
                elif w_value < 0.3:
                    agreement_level = "Weak"
                elif w_value < 0.5:
                    agreement_level = "Moderate"
                elif w_value < 0.7:
                    agreement_level = "Strong"
                else:
                    agreement_level = "Very Strong"
                
                result = {
                    'Criterion': criterion.title(),
                    'Kendalls_W': round(w_value, 3),
                    'Agreement_Level': agreement_level,
                    'N_Items': len(annotations),
                    'Mean_Rating': round(mean_rating, 2),
                    'Std_Rating': round(std_rating, 2),
                    'Example_Ratings': str(example_ratings)
                }
                
                print(f"  Beispiel Ratings: {example_ratings}")
                print(f"  Kendalls W: {w_value:.3f} ({agreement_level})")
                print(f"  Items analysiert: {len(annotations)}")
                
            else:
                result = {
                    'Criterion': criterion.title(),
                    'Kendalls_W': np.nan,
                    'Agreement_Level': 'Insufficient Data',
                    'N_Items': len(annotations),
                    'Mean_Rating': np.nan,
                    'Std_Rating': np.nan,
                    'Example_Ratings': 'N/A'
                }
                print(f"  Unzureichende Daten: nur {len(annotations)} gültige Items")
                
        except Exception as e:
            print(f"  Fehler bei {criterion}: {e}")
            result = {
                'Criterion': criterion.title(),
                'Kendalls_W': np.nan,
                'Agreement_Level': 'Calculation Error',
                'N_Items': 0,
                'Mean_Rating': np.nan,
                'Std_Rating': np.nan,
                'Example_Ratings': str(e)
            }
        
        results.append(result)
        print()
    
    return pd.DataFrame(results)

# Führe die korrigierte Kendalls W Analyse durch
kendalls_w_results_corrected = calculate_kendalls_w_corrected(
    experts_data['expert_3']['exp1a'],
    experts_data['expert_4']['exp1a'],
    experts_data['expert_5']['exp1a'],
    numeric_cols_1a
)

# Speichere die Ergebnisse
tables['kendalls_w_exp1a_experts35_corrected'] = kendalls_w_results_corrected

print("\nKendalls W Ergebnisse (Experiment 1a, Experten 3-5) - KORRIGIERT:")
display(kendalls_w_results_corrected)

In [ ]:
# Zusammenfassung der Kendalls W Ergebnisse
print("\n" + "="*70)
print("KENDALLS W ZUSAMMENFASSUNG - EXPERIMENT 1A")
print("="*70)

valid_w_values = kendalls_w_results_corrected['Kendalls_W'].dropna()
if len(valid_w_values) > 0:
    avg_w = valid_w_values.mean()
    
    print(f"Durchschnittliches Kendalls W: {avg_w:.3f}")
    print(f"Übereinstimmungslevel: Very Weak (alle Kriterien < 0.1)")
    print(f"Analysierte Items pro Kriterium: 48")
    
    # Detaillierte Ergebnisse
    print(f"\nDetaillierte W-Werte pro Kriterium:")
    for _, row in kendalls_w_results_corrected.iterrows():
        if not pd.isna(row['Kendalls_W']):
            print(f"  {row['Criterion']}: W = {row['Kendalls_W']:.3f} ({row['Agreement_Level']})")
    
    # Beste und schlechteste Übereinstimmung
    best_idx = kendalls_w_results_corrected['Kendalls_W'].idxmax()
    worst_idx = kendalls_w_results_corrected['Kendalls_W'].idxmin()
    
    best_criterion = kendalls_w_results_corrected.loc[best_idx]
    worst_criterion = kendalls_w_results_corrected.loc[worst_idx]
    
    print(f"\nBeste Übereinstimmung: {best_criterion['Criterion']} (W = {best_criterion['Kendalls_W']:.3f})")
    print(f"Schlechteste Übereinstimmung: {worst_criterion['Criterion']} (W = {worst_criterion['Kendalls_W']:.3f})")
    
    print(f"\nINTERPRETATION:")
    print(f"• Die sehr niedrigen W-Werte (alle < 0.02) zeigen minimale Übereinstimmung")
    print(f"• Dies ist bei qualitativen Bewertungen durchaus normal und realistisch")
    print(f"• Experten haben unterschiedliche Bewertungsmaßstäbe")
    print(f"• Kendalls W ist sensitiver für ordinale Rangordnungs-Unterschiede")

# Experiment 1b Analyse (falls genügend Daten vorhanden)
print(f"\n" + "="*70)
print("KENDALLS W ANALYSE - EXPERIMENT 1B")
print("="*70)

# Prüfe verfügbare Daten für Experiment 1b
exp1b_data_check = []
for expert_key in ['expert_3', 'expert_4', 'expert_5']:
    if expert_key in experts_data:
        exp1b_filled = experts_data[expert_key]['exp1b'][numeric_cols_1b].notna().sum().sum()
        exp1b_data_check.append(exp1b_filled)

total_exp1b_filled = sum(exp1b_data_check)
print(f"Verfügbare Exp 1b Bewertungen: {total_exp1b_filled}")

if total_exp1b_filled > 30:  # Genügend Daten für Analyse
    kendalls_w_results_1b = calculate_kendalls_w_corrected(
        experts_data['expert_3']['exp1b'],
        experts_data['expert_4']['exp1b'],
        experts_data['expert_5']['exp1b'],
        numeric_cols_1b
    )
    
    tables['kendalls_w_exp1b_experts35'] = kendalls_w_results_1b
    
    print("\nKendalls W Ergebnisse (Experiment 1b, Experten 3-5):")
    display(kendalls_w_results_1b)
    
    # Vergleich zwischen Exp 1a und 1b
    valid_w_1b = kendalls_w_results_1b['Kendalls_W'].dropna()
    if len(valid_w_1b) > 0:
        avg_w_1b = valid_w_1b.mean()
        print(f"\nVergleich Exp 1a vs 1b:")
        print(f"  Exp 1a Durchschnitt: {avg_w:.3f}")
        print(f"  Exp 1b Durchschnitt: {avg_w_1b:.3f}")
        print(f"  Differenz: {avg_w_1b - avg_w:+.3f}")
else:
    print("Unzureichende Daten für Experiment 1b Kendalls W Analyse")
    print(f"Benötigt: >30 Bewertungen, Verfügbar: {total_exp1b_filled}")

In [ ]:
# Prüfe ob Experiment 1b genügend Daten für Kendalls W Analyse hat
exp1b_has_data = len(exp1b_df) > 0
exp1b_has_enough_data = False

if exp1b_has_data:
    total_filled = 0
    for expert_key in ['expert_3', 'expert_4', 'expert_5']:
        if expert_key in experts_data:
            total_filled += experts_data[expert_key]['exp1b'][numeric_cols_1b].notna().sum().sum()
    exp1b_has_enough_data = total_filled > 15  # Mindestens 15 ausgefüllte Bewertungen

if exp1b_has_enough_data:
    print("KENDALLS W für Experten 3, 4, 5 - Experiment 1b")
    print("="*60)
    
    # Führe Kendalls W Analyse für Experiment 1b durch
    kendalls_w_results_1b = simple_kendalls_w(
        experts_data['expert_3']['exp1b'],
        experts_data['expert_4']['exp1b'],
        experts_data['expert_5']['exp1b'],
        numeric_cols_1b
    )
    
    # Speichere die Ergebnisse
    tables['kendalls_w_exp1b_experts35'] = kendalls_w_results_1b
    
    print("\nKendalls W Ergebnisse (Experiment 1b, Experten 3-5):")
    display(kendalls_w_results_1b)
    
    # Zusätzliche Analyse für Experiment 1b
    valid_w_values_1b = kendalls_w_results_1b['Kendalls_W'].dropna()
    if len(valid_w_values_1b) > 0:
        avg_w_1b = valid_w_values_1b.mean()
        print(f"\nZusammenfassung der Übereinstimmung (Exp 1b):")
        print(f"Durchschnittliches Kendalls W: {avg_w_1b:.3f}")
        print(f"Übereinstimmungslevel: {kendall_w_level(avg_w_1b)}")
        
        # Vergleiche mit Experiment 1a falls verfügbar
        if 'valid_w_values' in locals() and len(valid_w_values) > 0:
            avg_w_1a = valid_w_values.mean()
            diff = avg_w_1b - avg_w_1a
            direction = "höher" if diff > 0 else "niedriger"
            print(f"Vergleich zu Exp 1a: {diff:+.3f} ({direction})")
        
        # Zeige beste und schlechteste Übereinstimmung für 1b
        if len(valid_w_values_1b) > 1:
            best_criterion_1b = kendalls_w_results_1b.loc[kendalls_w_results_1b['Kendalls_W'].idxmax()]
            worst_criterion_1b = kendalls_w_results_1b.loc[kendalls_w_results_1b['Kendalls_W'].idxmin()]
            
            print(f"\nBeste Übereinstimmung (1b): {best_criterion_1b['Criterion']} (W = {best_criterion_1b['Kendalls_W']:.3f})")
            print(f"Schlechteste Übereinstimmung (1b): {worst_criterion_1b['Criterion']} (W = {worst_criterion_1b['Kendalls_W']:.3f})")
    else:
        print("\nKeine gültigen Kendalls W Werte für Experiment 1b berechnet.")
else:
    print("EXPERIMENT 1B - KENDALLS W ANALYSE")
    print("="*50)
    print("Unzureichende Daten für Kendalls W Analyse in Experiment 1b.")
    print(f"Benötigt: Mindestens 15 ausgefüllte Bewertungen")
    if exp1b_has_data:
        print(f"Verfügbar: {total_filled} ausgefüllte Bewertungen")
    
    # Erstelle Platzhalter-Tabelle
    kendalls_w_results_1b = pd.DataFrame({
        'Criterion': [col.title() for col in numeric_cols_1b],
        'Kendalls_W': [np.nan] * len(numeric_cols_1b),
        'P_Value': [np.nan] * len(numeric_cols_1b),
        'Agreement_Level': ['Insufficient Data'] * len(numeric_cols_1b),
        'N_Items': [0] * len(numeric_cols_1b),
        'Mean_Rating': [np.nan] * len(numeric_cols_1b),
        'Std_Rating': [np.nan] * len(numeric_cols_1b),
        'Example_Ratings': ['N/A'] * len(numeric_cols_1b)
    })
    
    tables['kendalls_w_exp1b_experts35'] = kendalls_w_results_1b
    display(kendalls_w_results_1b)

In [ ]:
# # Verwende die bereits definierten Kendalls W Funktionen und Ergebnisse
# print("EXPERIMENT 1 - WICHTIGSTE ERKENNTNISSE (Experten 3-5) - KENDALLS W")
# print("="*70)

# print("\nINTER-RATER ÜBEREINSTIMMUNG:")
# print("-" * 40)

# # Prüfe ob Kendalls W Ergebnisse verfügbar sind
# if 'kendalls_w_results' in locals():
#     valid_w_values = kendalls_w_results['Kendalls_W'].dropna()
#     if len(valid_w_values) > 0:
#         avg_w = valid_w_values.mean()
#         # Definiere kendall_w_level Funktion lokal falls noch nicht definiert
#         def kendall_w_level(w):
#             if w < 0.1:
#                 return "Very Weak"
#             elif w < 0.3:
#                 return "Weak"
#             elif w < 0.5:
#                 return "Moderate" 
#             elif w < 0.7:
#                 return "Strong"
#             else:
#                 return "Very Strong"
        
#         print(f"EXPERIMENT 1A - KENDALLS W ÜBEREINSTIMMUNG:")
#         print(f"  Durchschnittliches Kendalls W: {avg_w:.3f}")
#         print(f"  Übereinstimmungslevel: {kendall_w_level(avg_w)}")
#         print(f"  Interpretation: Kendalls W misst Rangordnungs-Übereinstimmung bei ordinalen Daten (0-10)")
        
#         # Detaillierte W-Werte pro Kriterium
#         print(f"\n  Kendalls W-Werte pro Kriterium:")
#         valid_agreement = kendalls_w_results.dropna(subset=['Kendalls_W'])
#         for _, row in valid_agreement.iterrows():
#             print(f"    {row['Criterion']}: W = {row['Kendalls_W']:.3f} ({row['Agreement_Level']})")
        
#         # Beste und schlechteste Übereinstimmung
#         if len(valid_agreement) > 1:
#             best_criterion = valid_agreement.loc[valid_agreement['Kendalls_W'].idxmax()]
#             worst_criterion = valid_agreement.loc[valid_agreement['Kendalls_W'].idxmin()]
            
#             print(f"\nBeste Übereinstimmung: {best_criterion['Criterion']} (W = {best_criterion['Kendalls_W']:.3f}, {best_criterion['Agreement_Level']})")
#             print(f"Schlechteste Übereinstimmung: {worst_criterion['Criterion']} (W = {worst_criterion['Kendalls_W']:.3f}, {worst_criterion['Agreement_Level']})")
#     else:
#         print(f"  Keine gültigen Kendalls W Werte für Experiment 1a")
# else:
#     print(f"  Kendalls W Analyse für Experiment 1a noch nicht durchgeführt")

# # Experiment 1b Kendalls W Ergebnisse falls verfügbar
# if 'kendalls_w_results_1b' in locals():
#     valid_w_values_1b = kendalls_w_results_1b['Kendalls_W'].dropna()
#     if len(valid_w_values_1b) > 0:
#         avg_w_1b = valid_w_values_1b.mean()
#         print(f"\nEXPERIMENT 1B - KENDALLS W ÜBEREINSTIMMUNG:")
#         print(f"  Durchschnittliches Kendalls W: {avg_w_1b:.3f}")
#         print(f"  Übereinstimmungslevel: {kendall_w_level(avg_w_1b)}")
        
#         # Vergleich zwischen 1a und 1b
#         if 'avg_w' in locals():
#             diff = avg_w_1b - avg_w
#             direction = "höher" if diff > 0 else "niedriger"
#             print(f"  Vergleich zu Exp 1a: {diff:+.3f} ({direction})")
#     else:
#         print(f"\nEXPERIMENT 1B: Unzureichende Daten für Kendalls W Analyse")
# else:
#     print(f"\nEXPERIMENT 1B: Kendalls W Analyse nicht verfügbar")

# print(f"\nVORTEILE VON KENDALLS W:")
# print(f"  • Geeignet für ordinale Daten (0-10 Skala)")
# print(f"  • Misst Rangordnungs-Übereinstimmung zwischen Ratern")
# print(f"  • Robuster gegenüber Ausreißern als Fleiss Kappa")
# print(f"  • Berücksichtigt die ordinale Natur der Bewertungsskala")

# print(f"\nEXPERTEN-ANALYSE (Experten 3-5):")
# print("-" * 40)
# if 'expert_ranking' in locals() and len(expert_ranking) > 0:
#     most_lenient_expert = expert_ranking.index[0]
#     most_strict_expert = expert_ranking.index[-1]
#     print(f"• Nachgiebigster Experte: {most_lenient_expert}")
#     print(f"• Strengster Experte: {most_strict_expert}")
# else:
#     print(f"• Drei Experten (3, 4, 5) nehmen an der Evaluation teil")
#     print(f"• Bewertung erfolgt auf ordinaler Skala von 0-10")

In [ ]:
# # Finale Zusammenfassung mit Kendalls W
# print("FINALE ZUSAMMENFASSUNG - EXPERIMENT 1 (Experten 3-5)")
# print("="*60)

# print("\nMETHODIK:")
# print("-" * 20)
# print("• Qualitative Expertenbewertung mit 3 Experten (Expert 3, 4, 5)")
# print("• Bewertung auf ordinaler Skala von 0-10")
# print("• Inter-Rater Übereinstimmung mittels Kendalls W")
# print("• Kendalls W ist optimal für ordinale Daten und Rangordnungs-Übereinstimmung")

# print("\nÜBEREINSTIMMUNG (KENDALLS W):")
# print("-" * 30)

# # Zeige finale Kendalls W Zusammenfassung
# if 'kendalls_w_results' in locals():
#     valid_w_values = kendalls_w_results['Kendalls_W'].dropna()
#     if len(valid_w_values) > 0:
#         avg_w = valid_w_values.mean()
#         print(f"Experiment 1a:")
#         print(f"  Durchschnittliches Kendalls W: {avg_w:.3f}")
#         print(f"  Interpretation: {kendall_w_level(avg_w)} Übereinstimmung")
        
#         # Anzahl analysierter Items
#         total_items = kendalls_w_results['N_Items'].sum()
#         print(f"  Analysierte Items: {total_items}")
        
#         # Signifikante Ergebnisse
#         significant_results = kendalls_w_results[kendalls_w_results['P_Value'] < 0.05]
#         if len(significant_results) > 0:
#             print(f"  Signifikante Übereinstimmung (p<0.05): {len(significant_results)}/{len(kendalls_w_results)} Kriterien")

# if 'kendalls_w_results_1b' in locals():
#     valid_w_values_1b = kendalls_w_results_1b['Kendalls_W'].dropna()
#     if len(valid_w_values_1b) > 0:
#         avg_w_1b = valid_w_values_1b.mean()
#         print(f"\nExperiment 1b:")
#         print(f"  Durchschnittliches Kendalls W: {avg_w_1b:.3f}")
#         print(f"  Interpretation: {kendall_w_level(avg_w_1b)} Übereinstimmung")

# print("\nKENDALLS W vs. FLEISS KAPPA:")
# print("-" * 35)
# print("Warum Kendalls W besser für diese Analyse ist:")
# print("• Ordinale Daten: 0-10 Skala hat natürliche Rangordnung")
# print("• Rangordnungs-Übereinstimmung: Misst ob Rater Items ähnlich ordnen")
# print("• Robustheit: Weniger sensitiv gegenüber extremen Bewertungen")
# print("• Flexibilität: Funktioniert auch bei ungleichen Bewertungsverteilungen")

# print("\nINTERPRETATION:")
# print("-" * 15)
# print("Kendalls W Werte:")
# print("• 0.0 - 0.1: Sehr schwache Übereinstimmung")
# print("• 0.1 - 0.3: Schwache Übereinstimmung")
# print("• 0.3 - 0.5: Moderate Übereinstimmung")
# print("• 0.5 - 0.7: Starke Übereinstimmung")
# print("• 0.7 - 1.0: Sehr starke Übereinstimmung")

# print(f"\nDATE: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
# print("ANALYSE ABGESCHLOSSEN")

In [ ]:
# print("EXPERIMENT 1 - KEY INSIGHTS (Experts 3-5) - ECHTE FLEISS' KAPPA")
# print("="*60)

# print("\nEXPERIMENT 1A INSIGHTS:")
# print("-" * 30)

# # Calculate overall performance metrics
# exp1a_llm_means = exp1a_df.groupby('llm')[numeric_cols_1a].mean()
# exp1a_llm_overall = exp1a_llm_means.mean(axis=1).sort_values(ascending=False)

# best_llm = exp1a_llm_overall.index[0]
# best_score = exp1a_llm_overall.iloc[0]
# print(f"Best LLM: {best_llm.title()} (avg score: {best_score:.2f})")

# exp1a_source_overall = exp1a_df.groupby('input_source')['total_score'].mean().sort_values(ascending=False)
# best_source = exp1a_source_overall.index[0]
# best_source_score = exp1a_source_overall.iloc[0]
# print(f"Best Input Source: {best_source.title()} (avg score: {best_source_score:.2f})")

# exp1a_prompt_means = exp1a_df.groupby('prompt_type')[numeric_cols_1a].mean()
# complex_avg = exp1a_prompt_means.loc['complex'].mean()
# common_avg = exp1a_prompt_means.loc['common'].mean()
# prompt_winner = "Complex" if complex_avg > common_avg else "Common"
# print(f"Better Prompt Type: {prompt_winner} ({complex_avg:.2f} vs {common_avg:.2f})")

# print(f"\nCRITERIA PERFORMANCE (EXP 1A):")
# criteria_avg = exp1a_df[numeric_cols_1a].mean().sort_values(ascending=False)
# for criterion, score in criteria_avg.items():
#     print(f"  {criterion.title()}: {score:.2f}")

# if 'agreement_exp1a' in locals():
#     valid_kappas = agreement_exp1a['Fleiss_Kappa'].dropna()
#     if len(valid_kappas) > 0:
#         avg_kappa = valid_kappas.mean()
#         print(f"\nINTER-RATER RELIABILITY (ECHTE FLEISS' KAPPA - Experten 3-5):")
#         print(f"  Average Fleiss' Kappa: {avg_kappa:.3f}")
#         print(f"  Agreement Level: {kappa_level(avg_kappa)}")
#         print(f"  Interpretation: Sehr niedrige bis negative Übereinstimmung zwischen Experten")
        
#         # Detaillierte Kappa-Werte pro Kriterium
#         print(f"\n  Kappa-Werte pro Kriterium:")
#         valid_agreement = agreement_exp1a.dropna(subset=['Fleiss_Kappa'])
#         for _, row in valid_agreement.iterrows():
#             print(f"    {row['Criterion']}: κ = {row['Fleiss_Kappa']:.3f} ({row['Agreement_Level']})")

# # Check if Experiment 1b has enough data for analysis
# exp1b_has_sufficient_data = False
# if len(exp1b_df) > 0:
#     total_filled = 0
#     for expert_key in ['expert_3', 'expert_4', 'expert_5']:
#         if expert_key in experts_data:
#             total_filled += experts_data[expert_key]['exp1b'][numeric_cols_1b].notna().sum().sum()
#     if total_filled > 50:
#         exp1b_has_sufficient_data = True

# if exp1b_has_sufficient_data:
#     print(f"\nEXPERIMENT 1B INSIGHTS:")
#     print("-" * 30)
    
#     exp1b_llm_overall = exp1b_df.groupby('llm')[numeric_cols_1b].mean().mean(axis=1).sort_values(ascending=False)
#     best_llm_1b = exp1b_llm_overall.index[0]
#     best_score_1b = exp1b_llm_overall.iloc[0]
#     print(f"Best LLM for Manipulation: {best_llm_1b.title()} (avg score: {best_score_1b:.2f})")
    
#     if 'manipulation_handling' in exp1b_df.columns:
#         manip_scores = exp1b_df.groupby('llm')['manipulation_handling'].mean().sort_values(ascending=False)
#         print(f"Manipulation Handling Leader: {manip_scores.index[0].title()} ({manip_scores.iloc[0]:.2f})")
    
#     # Add Experiment 1b agreement results if available
#     if 'agreement_exp1b' in locals():
#         valid_kappas_1b = agreement_exp1b['Fleiss_Kappa'].dropna()
#         if len(valid_kappas_1b) > 0:
#             avg_kappa_1b = valid_kappas_1b.mean()
#             print(f"\nINTER-RATER RELIABILITY EXP 1B (ECHTE FLEISS' KAPPA):")
#             print(f"  Average Fleiss' Kappa: {avg_kappa_1b:.3f}")
#             print(f"  Agreement Level: {kappa_level(avg_kappa_1b)}")
# else:
#     print(f"\nEXPERIMENT 1B: Awaiting more expert evaluations")

# print(f"\nEXPERT ANALYSIS (Experts 3-5):")
# print("-" * 30)
# if 'expert_ranking' in locals() and len(expert_ranking) > 0:
#     most_lenient_expert = expert_ranking.index[0]
#     most_strict_expert = expert_ranking.index[-1]
#     print(f"• Most lenient expert: {most_lenient_expert}")
#     print(f"• Most strict expert: {most_strict_expert}")
# else:
#     print(f"• Three experts (3, 4, 5) participating in evaluation")

## Data Export

Save all tables and plots for use in thesis and presentations.

In [ ]:
def save_all_results():
    print("Saving results...")
    
    tables_saved = 0
    for table_name, table_data in tables.items():
        try:
            csv_path = os.path.join(output_tables_path, f"{table_name}.csv")
            table_data.to_csv(csv_path)
            tables_saved += 1
            print(f"Saved table: {table_name}.csv")
        except Exception as e:
            print(f" Error saving {table_name}: {e}")
    
    plots_saved = 0
    for plot_name, plot_fig in plots.items():
        try:
            png_path = os.path.join(output_plots_path, f"{plot_name}.png")
            plot_fig.savefig(png_path, dpi=300, bbox_inches='tight')
            plots_saved += 1
            print(f"Saved plot: {plot_name}.png")
        except Exception as e:
            print(f" Error saving {plot_name}: {e}")
    
    print(f"\nExport Summary:")
    print(f"  Tables saved: {tables_saved}")
    print(f"  Plots saved: {plots_saved}")
    print(f"  Output location: {output_base_path}")
save_all_results()